# Thesis pipeline server — Qwen3.6-27B Q6_K (MTP build) + speculative decoding

`llama-server` (GGUF) + Cloudflare tunnel, sized for the **fiche + note**
pipelines. Pair with `scripts/run_fiche.py` and
`scripts/run_note.py` running on your Mac.

Model is fixed: **Qwen3.6-27B, Q6_K, MTP build** (`unsloth/Qwen3.6-27B-MTP-GGUF`),
served with **`--spec-type draft-mtp`** — llama.cpp drafts from the model's own
MTP head (`blk.64.nextn.*`), verified by the 27B: same output distribution,
decode 55.7 → ~107 tok/s on prose / ~150 on verbatim copy (promoted 2026-08-17,
progress_notes §13). **3 slots** (v3, §14) so the note loop can run with rlm
`truncate_len 20000`. v3 per thesis ≈ 4.5–5 min (Bourse ~305 s, Daley ~275 s).
The pre-MTP file (`unsloth/Qwen3.6-27B-GGUF`, same filename!) has NO head — keep it
in its own Drive dir if you need to roll back (`SPEC_EXTRA_ARGS = []` +
`GGUF_REPO/GGUF_LOCAL_DIR` back to the old values).

Workflow:
1. Run all cells **in order**.
2. The tunnel cell prints a `https://xxxx.trycloudflare.com` URL.
3. On your Mac:
   ```bash
   export QWEN_SERVER_URL=https://xxxx.trycloudflare.com/v1
   export QWEN_API_KEY=colab
   export SHIM_MODEL_NAME=Qwen3.6-27B-Q6_K.gguf
   python3 scripts/run_shim.py &
   ```
4. Then run the fiche steps, audit, and the note. See the repo README.


## 0. Configuration — edit this cell before running anything else

In [ ]:
# ── Model — FIXED for the thesis pipeline ────────────────────────────────────
# Qwen3.6-27B Q6_K is the quant all validated results were measured on.
# Changing it invalidates the gold comparison in gold/.
GGUF_REPO  = 'unsloth/Qwen3.6-27B-MTP-GGUF'   # MTP build (carries blk.64.nextn.*). Pre-MTP: 'unsloth/Qwen3.6-27B-GGUF'
GGUF_NAME  = 'Qwen3.6-27B'
QUANT      = 'Q6_K'
# ⚠ Both repos publish the SAME filename (Qwen3.6-27B-Q6_K.gguf). They live in
# different Drive dirs (GGUF_LOCAL_DIR below) so one can never overwrite the other.

# ── Speculative decoding (promoted 2026-08-17) ───────────────────────────────
# Appended verbatim to the launch command. The launch cell asserts the GGUF has
# the MTP head when 'draft-mtp' is requested. [] = pre-MTP behaviour.
SPEC_EXTRA_ARGS = ['--spec-type', 'draft-mtp']

# ── Inference ────────────────────────────────────────────────────────────────
REASONING_BUDGET = -1       # thinking cap: -1 = unlimited. Callers set their own.
MAX_TOKENS       = 4096     # server-side DEFAULT only; every caller overrides it
N_GPU_LAYERS     = 99       # 99 = all layers on GPU

# ── Context sizing ───────────────────────────────────────────────────────────
# --ctx-size is the TOTAL and is SPLIT EVENLY across slots:
#     per_slot = CTX_SIZE // N_PARALLEL
#
# MEASURED per-request peaks (prompt + max output, at 2.85 chars/token
# calibrated from real usage data — NOT the len//4 the scripts print):
#     Bourse step2b conclusion .. 47,753   <- BINDING CONSTRAINT
#     Daley  step3b concepts .... 44,807
#     Bourse step3b concepts .... 39,250
#     Bourse RLM note (peak) .... 35,341
#
# N_PARALLEL=3 (promoted 2026-08-19, §14): 262144/3 = 87,381 tokens per slot. The note
# loop is single-slot and, with rlm truncate_len=20000, peaks at ~45-55k tokens (65k was
# too tight for long loops); the fiche's 3a∥3b uses two slots; rlm fan-out is unused.
N_PARALLEL = 3
CTX_SIZE   = 262144         # -> 87,381 per slot = 1.83x the 47,753 peak

# KV quantisation is REQUIRED at this context on a 96 GB card:
#   f16  -> 262144 x 256 KB = 68.7G KV + 22G model + ~8G buffers = 98.7G  OOM
#   q8_0 -> 262144 x 128 KB = 34.4G KV + 22G model + ~8G buffers = 64.4G  fits
# NB the champion runs used f16 (llama.cpp default — no --cache-type was passed).
# This is a deliberate departure: q8_0 KV is ~lossless, running out of slot
# context mid-run is not. To stay strictly faithful instead, use
# N_PARALLEL=3 / CTX_SIZE=196608 / 'f16' (same 65,536 per slot, 81G total).
KV_TYPE = 'q8_0'

BATCH  = 8192               # prompt-processing logical batch
UBATCH = 2048               # physical micro-batch — large win on 12k-41k prompts

# ── Sampling ─────────────────────────────────────────────────────────────────
# These are server-side DEFAULTS. The fiche script sends temp/top_p/top_k per
# call, so they apply only to callers that omit them — i.e. rlm-cli, whose
# request is just model/messages/stream/stream_options/max_completion_tokens.
# Control the note pipeline via the shim: SHIM_TEMPERATURE / SHIM_TOP_K / SHIM_TOP_P.
TEMPERATURE      = 1.0
TOP_P            = 0.95
TOP_K            = 20
MIN_P            = 0.0
#PRESENCE_PENALTY = 1.5     # left OFF (llama.cpp default 0.0). The Bourse
                            # champion ran with 1.5, the Daley champion with 0.0.

# ── Server ───────────────────────────────────────────────────────────────────
PORT    = 8000
API_KEY = 'colab'           # same value as QWEN_API_KEY on your Mac

# ── Paths ────────────────────────────────────────────────────────────────────
DRIVE_MODELS_DIR = '/content/drive/MyDrive/hf_models'
LLAMA_BUILD_DIR  = '/content/llama.cpp/build'

# Derived (do not edit)
import os
GGUF_FILENAME  = f'{GGUF_NAME}-{QUANT}.gguf'
GGUF_LOCAL_DIR = os.path.join(DRIVE_MODELS_DIR, 'qwen36-mtp-gguf' if 'MTP' in GGUF_REPO else 'qwen36-gguf')
GGUF_PATH      = os.path.join(GGUF_LOCAL_DIR, GGUF_FILENAME)

per_slot = CTX_SIZE // N_PARALLEL
assert per_slot >= 48000, f'per-slot ctx {per_slot:,} too small: fiche step2b needs ~47,753'
est_kv = CTX_SIZE * (128*1024 if KV_TYPE == 'q8_0' else 256*1024) / 1e9
print(f'Model     : {GGUF_PATH}')
print(f'Spec      : {SPEC_EXTRA_ARGS or "OFF"}')
print(f'Slots     : {N_PARALLEL}  total_ctx={CTX_SIZE:,}  per_slot={per_slot:,} '
      f'({per_slot/47753:.2f}x the measured peak)')
print(f'KV        : {KV_TYPE}  ~{est_kv:.1f} GB estimated  (+22 GB model +~8 GB buffers '
      f'= ~{est_kv+30:.1f} GB)')
print(f'Sampling  : temp={TEMPERATURE}, top_p={TOP_P}, top_k={TOP_K}, min_p={MIN_P}')
print()
print('VERIFY the KV estimate after load — it assumes 64 layers / 8 KV heads / 128 dim:')
print("  !grep -iE 'KV self size|n_ctx_per_seq|flash.attn' /content/llama_server.log")


## 1. Build llama.cpp with CUDA

Compiles `llama-server` from the latest master.
Takes ~5 min on first run. The binary **is** cached to Drive (keyed by GPU compute capability).

In [ ]:
import subprocess, hashlib, shutil

def run(cmd, **kwargs):
    print(f'$ {cmd}')
    subprocess.check_call(cmd, shell=True, **kwargs)

run('apt-get install -y -q cmake ninja-build')

# Mount Drive early so we can cache the compiled binary (GPU-arch specific).
# Safe to call even if already mounted.
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)

import os
# Delete the stale cached binary + local copy
!rm -f {BINARY_CACHE} /content/llama-server
# Also wipe the build dir so cmake starts fresh
!rm -rf /content/llama.cpp/build

# Use GPU compute capability to key the cached binary
gpu_arch = subprocess.check_output(
    'nvidia-smi --query-gpu=compute_cap --format=csv,noheader',
    shell=True
).decode().strip().replace('.', '')

BINARY_CACHE = os.path.join(DRIVE_MODELS_DIR, f'llama-server-sm{gpu_arch}')
SERVER_BIN   = '/content/llama-server'

if os.path.isfile(BINARY_CACHE):
    shutil.copy(BINARY_CACHE, SERVER_BIN)
    os.chmod(SERVER_BIN, 0o755)
    print(f'✅ Loaded cached binary (sm{gpu_arch}) — skipping build')
else:
    print(f'No cached binary for sm{gpu_arch}, building from source (~5 min)...')
    if not os.path.isdir('/content/llama.cpp'):
        run('git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp')
    run(
        'cmake -S /content/llama.cpp -B /content/llama.cpp/build '
        '-DGGML_CUDA=ON '
        '-DCMAKE_CUDA_ARCHITECTURES=native '
        '-DCMAKE_BUILD_TYPE=Release '
        '-GNinja'
    )
    run('cmake --build /content/llama.cpp/build --target llama-server -j$(nproc)')
    # Cache the freshly built binary to Drive for next session
    shutil.copy('/content/llama.cpp/build/bin/llama-server', BINARY_CACHE)
    shutil.copy('/content/llama.cpp/build/bin/llama-server', SERVER_BIN)
    os.chmod(SERVER_BIN, 0o755)
    print(f'✅ Binary built and cached to Drive for next time')

run(f'{SERVER_BIN} --version')

## 2. Hugging Face login (optional for unsloth public GGUFs, but raises rate limits)

In [ ]:
# Never paste tokens here. Prefer Colab secrets: from google.colab import userdata; login(userdata.get('HF_TOKEN'))
from huggingface_hub import login
login()  # paste your HF token when prompted — or skip if already authenticated

## 3. Mount Drive + download GGUF

The GGUF is cached on Drive after the first download (~22 GB for Q6_K).
Subsequent runs skip the download and load from Drive directly.

In [ ]:
# Drive is already mounted in cell above; this just ensures the model subdir exists
os.makedirs(GGUF_LOCAL_DIR, exist_ok=True)

if os.path.isfile(GGUF_PATH):
    size_gb = os.path.getsize(GGUF_PATH) / 1e9
    print(f'✅ GGUF already cached ({size_gb:.1f} GB): {GGUF_PATH}')
else:
    print(f'Downloading {GGUF_REPO} [{GGUF_FILENAME}] → {GGUF_LOCAL_DIR}')
    from huggingface_hub import hf_hub_download
    hf_hub_download(
        repo_id=GGUF_REPO,
        filename=GGUF_FILENAME,
        local_dir=GGUF_LOCAL_DIR,
    )
    print(f'✅ Download complete: {GGUF_PATH}')

run('nvidia-smi')

## 4. Start llama-server

Key flags for Qwen3.6:
- `--jinja` — enables the Qwen3.6 Jinja chat template (required for thinking tokens)
- `--reasoning-budget N` — hard-caps thinking to N tokens (-1 = unlimited)
- `--chat-template-kwargs` — toggles thinking globally (can be overridden per request)
- `--special` — keeps special tokens in the stream (not needed with llama-server's native `reasoning_content` separation)
- `--n-gpu-layers 99` — offloads all layers to GPU
- `--ctx-size` — total KV cache window
- `--n-predict` — default max output tokens (overridable per request via `max_tokens`)

The server exposes an OpenAI-compatible `/v1/chat/completions` endpoint.
Thinking content is returned in `message.reasoning_content` (separate from `message.content`).
- `--spec-type draft-mtp` (via `SPEC_EXTRA_ARGS`) — MTP speculative decoding, promoted 2026-08-17; lossless, ~+90% decode


In [ ]:
import subprocess

LOG_PATH = '/content/llama_server.log'

per_slot = CTX_SIZE // N_PARALLEL
print(f'slots={N_PARALLEL}  total_ctx={CTX_SIZE:,}  per_slot={per_slot:,}  kv={KV_TYPE}')
assert per_slot >= 48000, f'per-slot ctx {per_slot:,} < 47,753 -> fiche step2b/3b would truncate'
print()
subprocess.run('nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv', shell=True)
subprocess.run('nvcc --version | tail -2', shell=True)   # Blackwell sm_120 needs CUDA >= 12.8
print()

# Guard: 'draft-mtp' needs the MTP head in THIS file (both unsloth repos share the filename).
if any('draft-mtp' in a for a in SPEC_EXTRA_ARGS):
    try:
        import gguf
    except ImportError:
        subprocess.run('pip install -q gguf', shell=True); import gguf
    _nextn = [t.name for t in gguf.GGUFReader(GGUF_PATH).tensors if 'nextn' in t.name]
    assert _nextn, (f'{GGUF_PATH} has NO blk.*.nextn.* tensors — this is the pre-MTP file. '
                    f'Point GGUF_REPO/GGUF_LOCAL_DIR at the MTP build or set SPEC_EXTRA_ARGS = [].')
    print(f'MTP head present: {len(_nextn)} nextn tensors ✓')

# Read the whole GGUF into memory up front instead of mmap-paging it off the Drive FUSE
# mount. llama.cpp spells this `--load-mode none` since 2026-09 (#28334 removed
# `--no-mmap`); older cached binaries only know `--no-mmap`. Pick whichever this
# binary accepts.
_help = subprocess.run([SERVER_BIN, '--help'], capture_output=True, text=True)
NO_MMAP_ARGS = ['--load-mode', 'none'] if '--load-mode' in (_help.stdout + _help.stderr) else ['--no-mmap']
print(f'no-mmap flag    : {" ".join(NO_MMAP_ARGS)}')

cmd = [
    SERVER_BIN,
    '--model',            GGUF_PATH,
    '--n-gpu-layers',     str(N_GPU_LAYERS),
    '--ctx-size',         str(CTX_SIZE),
    '--parallel',         str(N_PARALLEL),   # concurrent decode slots
    '--cont-batching',                       # continuous batching (default-on, explicit)
    '--flash-attn',       'on',              # large prefill win + less KV VRAM
    '--cache-type-k',     KV_TYPE,
    '--cache-type-v',     KV_TYPE,
    '--batch-size',       str(BATCH),
    '--ubatch-size',      str(UBATCH),
    *NO_MMAP_ARGS,                           # never page the GGUF off the Drive FUSE mount (see above)
    '--threads-http',     str(N_PARALLEL + 2),
    '--metrics',                             # /metrics endpoint for slot introspection
    '--slot-prompt-similarity', '0.85',  # route a request to the slot already holding its prefix (keeps the note loop's KV cache warm)
    '--n-predict',        str(MAX_TOKENS),
    '--reasoning-budget', str(REASONING_BUDGET),
    '--temp',             str(TEMPERATURE),
    '--top-p',            str(TOP_P),
    '--top-k',            str(TOP_K),
    '--min-p',            str(MIN_P),
#    '--presence-penalty', str(PRESENCE_PENALTY),
    '--jinja',                          # required for Qwen3.6 chat template
    '--chat-template-kwargs', '{"enable_thinking":true}',  # enable thinking by default
    '--special',
    '--port',             str(PORT),
    '--api-key',          API_KEY,
    '--log-file',         LOG_PATH,
] + list(SPEC_EXTRA_ARGS)          # speculative decoding (promoted 2026-08-17)

with open(LOG_PATH, 'w') as logf:
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)

print(f'llama-server PID : {proc.pid}')
print(f'Log              : {LOG_PATH}')
print(f'Reasoning budget : {REASONING_BUDGET} tokens')
print(f'Speculative      : {SPEC_EXTRA_ARGS or "OFF"}')
print(f'Loading model… {" ".join(NO_MMAP_ARGS)} reads the whole GGUF up front.')
print('Tail the log below until you see "server is listening".')
# NOTE: older llama.cpp builds take a bare `--flash-attn` with no value.
# If the server exits instantly on an arg-parse error, drop the 'on'.


In [ ]:
# Wait for the server to become ready (polls /health every 5 s, 10 min timeout).
# Stops at once if llama-server exits (bad flag, OOM, missing file) and shows why.
import time, requests, subprocess as _sp

ready = False
for i in range(120):
    time.sleep(5)
    if proc.poll() is not None:
        print(f'\n❌ llama-server exited with code {proc.returncode}. Last log lines:\n')
        print(_sp.run(f'tail -n 25 {LOG_PATH}', shell=True, capture_output=True, text=True).stdout)
        break
    print(_sp.run(f'tail -n 4 {LOG_PATH}', shell=True, capture_output=True, text=True).stdout + '-' * 60)
    try:
        r = requests.get(
            f'http://localhost:{PORT}/health',
            headers={'Authorization': f'Bearer {API_KEY}'},
            timeout=2,
        )
        if r.status_code == 200:
            print(f'\n✅ llama-server READY: {r.json()}')
            ready = True
            break
    except Exception:
        pass
else:
    print('\n⚠️  Server did not become ready in 10 min. Check the log above.')

if ready:
    print(_sp.run(f'grep -iE "spec|draft|mtp" {LOG_PATH} | tail -n 6', shell=True, capture_output=True, text=True).stdout)


## 5. Quick test

In [ ]:
from openai import OpenAI
import json

test_client = OpenAI(
    base_url=f'http://localhost:{PORT}/v1',
    api_key=API_KEY,
)

# Probe: which model ID does the server report?
models = test_client.models.list()
print('Available models:', [m.id for m in models.data])

# Use the first available model ID
MODEL_ID = models.data[0].id

resp = test_client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {'role': 'system', 'content': 'You are a concise assistant.'},
        {'role': 'user',   'content': 'Say hello in French in one short sentence.'},
    ],
    max_tokens=500,
    temperature=TEMPERATURE,
    extra_body={
        'thinking_budget_tokens': 500,
        'chat_template_kwargs': {'enable_thinking': True},
    },
)

print(json.dumps(json.loads(resp.model_dump_json()), indent=2))

## 6. Expose the server via Cloudflare tunnel

In [ ]:
# Install cloudflared (one-time per session)
if not os.path.exists('/usr/local/bin/cloudflared'):
    run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared')
    run('chmod +x /usr/local/bin/cloudflared')
print('cloudflared ready')

In [ ]:
# Launch the tunnel — keep this cell running for the duration of your session.
import subprocess, re, threading, time

tunnel_url = {'value': None}

def stream_tunnel():
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='')
        if tunnel_url['value'] is None:
            m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
            if m:
                tunnel_url['value'] = m.group(0)

t = threading.Thread(target=stream_tunnel, daemon=True)
t.start()

# Wait up to 30 s for the URL
for _ in range(30):
    if tunnel_url['value']:
        break
    time.sleep(1)

if tunnel_url['value']:
    print('\n' + '=' * 60)
    print(f'🎯 PUBLIC URL: {tunnel_url["value"]}   [spec: {SPEC_EXTRA_ARGS or "OFF"}]')
    print('=' * 60)
    print(f'\nOn your Mac, from the repo root (README "Quick start"):')
    print(f'  export QWEN_SERVER_URL={tunnel_url["value"]}/v1 QWEN_API_KEY={API_KEY} QWEN_MODEL={GGUF_FILENAME}')
    print(f'')
    print(f'  # fiche: straight to the tunnel (never through the shim, which overrides sampling)')
    print(f'  python3 scripts/run_fiche.py data/<thesis>.parquet --output-dir results/<thesis>')
    print(f'  python3 scripts/tools/verify_citations.py results/<thesis>/<thesis>_fiche.md data/<thesis>.parquet')
    print(f'')
    print(f'  # note: start the shim (it reads QWEN_SERVER_URL), then the note')
    print(f'  SHIM_MODEL_NAME={GGUF_FILENAME} SHIM_TEMPERATURE=0.6 SHIM_TOP_K=20 SHIM_TOP_P=0.95 \\')
    print(f'  SHIM_STABLE_PREFIX=1 SHIM_THINK_BUDGET=1500 SHIM_MAX_TOKENS=8192 python3 scripts/run_shim.py &')
    print(f'  python3 scripts/run_note.py --thesis <thesis> --fiche results/<thesis>/<thesis>_fiche.md --parquet data/<thesis>.parquet --model {GGUF_FILENAME}')
else:
    print('\n⚠️  Tunnel did not start. Check output above.')

## 6b. Throughput benchmark — is the GPU or the tunnel the bottleneck?

Reads llama.cpp's own `timings` block (authoritative, not estimated) and reports
**prefill** and **decode** tok/s for local vs tunnel, 1 vs N concurrent.

- `local N=k aggregate` ~= `local N=1`  -> server is **serialising** (`--parallel` too low)
- `local N=k aggregate` ~= 2.5-3.5x     -> continuous batching is working
- `tunnel N=1` ~= `local N=1`           -> tunnel is **not** the bottleneck
- `tunnel` markedly slower               -> Cloudflare quick-tunnel is throttling


In [ ]:
# ── Throughput benchmark ────────────────────────────────────────────────────
import time, statistics, requests
import concurrent.futures as cf

CONC = min(N_PARALLEL, 8)          # concurrency to test
LOCAL_BASE = f'http://localhost:{PORT}'
TUNNEL_BASE = None
try:
    if tunnel_url.get('value'):
        TUNNEL_BASE = tunnel_url['value']
except NameError:
    pass

HDRS = {'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'}
try:
    MODEL_ID
except NameError:
    MODEL_ID = requests.get(f'{LOCAL_BASE}/v1/models', headers=HDRS, timeout=30).json()['data'][0]['id']
print(f'model={MODEL_ID}  slots={N_PARALLEL}  per_slot={CTX_SIZE//N_PARALLEL:,}  testing N={CONC}')

FILLER = ('The cosmic microwave background is the oldest light in the universe, '
          'emitted at recombination roughly 380,000 years after the Big Bang. ') * 160  # ~3k tok

def one_call(base, max_tokens=256, think=False):
    payload = {
        'model': MODEL_ID,
        'messages': [{'role':'user','content': FILLER + '\n\nIn exactly 100 words, summarise the text above.'}],
        'max_tokens': max_tokens,
        'stream': False,
        'cache_prompt': False,                     # force a real prefill each time
        'chat_template_kwargs': {'enable_thinking': think},
    }
    t0 = time.time()
    r = requests.post(f'{base}/v1/chat/completions', headers=HDRS, json=payload, timeout=900)
    wall = time.time() - t0
    r.raise_for_status()
    t = r.json().get('timings') or {}
    return {'wall': wall, 'prompt_tps': t.get('prompt_per_second'),
            'pred_n': t.get('predicted_n'), 'pred_tps': t.get('predicted_per_second')}

def bench(base, n, label):
    t0 = time.time()
    if n == 1:
        res = [one_call(base)]
    else:
        with cf.ThreadPoolExecutor(max_workers=n) as ex:
            res = [f.result() for f in [ex.submit(one_call, base) for _ in range(n)]]
    wall = time.time() - t0
    per = statistics.mean([r['pred_tps'] or 0 for r in res])
    pre = statistics.mean([r['prompt_tps'] or 0 for r in res])
    agg_decode  = per * n          # <- the batching metric: prefill NOT in denominator
    agg_prefill = pre * n
    wallrate    = sum((r['pred_n'] or 0) for r in res) / wall if wall else 0
    print(f'  {label:<22} wall={wall:6.1f}s  decode/req={per:6.1f}  '
          f'AGG-DECODE={agg_decode:7.1f}  AGG-PREFILL={agg_prefill:8.1f}  (wall-rate {wallrate:5.1f})')
    return agg_decode

print('\n=== warm-up (not measured) ==='); one_call(LOCAL_BASE, max_tokens=16)
print('\n=== LOCAL (bypasses tunnel — raw GPU) ===')
l1 = bench(LOCAL_BASE, 1, 'N=1 sequential')
lk = bench(LOCAL_BASE, CONC, f'N={CONC} concurrent')

t1 = None
if TUNNEL_BASE:
    print(f'\n=== VIA TUNNEL ({TUNNEL_BASE}) ===')
    t1 = bench(TUNNEL_BASE, 1, 'N=1 sequential')
    bench(TUNNEL_BASE, CONC, f'N={CONC} concurrent')
else:
    print('\n(no tunnel URL in scope — run the tunnel cell first)')

print('\n' + '='*70); print('DIAGNOSIS'); print('='*70)
sp = lk/l1 if l1 else 0
print(f'  decode batching gain (local N={CONC} vs N=1): {sp:.2f}x')
print('    -> SERIALISED; --parallel is not taking effect' if sp < 1.3 else
      '    -> weak batching; check --cont-batching / --ubatch-size' if sp < 2.0 else
      '    -> continuous batching is working well.')
print()
print('  NOTE: compare AGG-DECODE, not the wall-rate. Wall-rate has prefill in')
print('  its denominator, so on a short-generation probe it understates batching')
print('  badly (we measured 1.73x wall-rate vs 3.76x true decode gain). The real')
print('  fiche/note workload is ~82% decode, so AGG-DECODE is the honest proxy.')
if t1:
    ratio = t1/l1 if l1 else 0
    print(f'  tunnel penalty (tunnel N=1 / local N=1):  {ratio:.2f}x')
    print('    -> tunnel is NOT the bottleneck.' if ratio > 0.85 else
          '    -> mild tunnel overhead.'          if ratio > 0.6  else
          '    -> TUNNEL IS THROTTLING; use a named tunnel or ssh -R.')
print('='*70)


## 7. (Optional) Monitor GPU + server while your pipeline runs

In [ ]:
import time
while True:
    !nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv,noheader
    !tail -n 3 {LOG_PATH}
    print('-' * 50)
    time.sleep(30)